# Speculative decoding benchmark for `generate_samples`

**Goal:** find out whether speculative decoding gives a *real* end-to-end speedup for the
kernel-generation workload in [`kernel_gen/generate_kernels_samples.py`](../kernel_gen/generate_kernels_samples.py),
and which proposer wins.

We compare four configs on the **same prompts, same sampling params**:

| Config | Proposer |
|---|---|
| `baseline` | none (plain decoding) |
| `draft Qwen3-1.7B` | small draft model |
| `draft Qwen3-0.6B` | smaller draft model |
| `ngram` | prompt-lookup (no draft model, zero VRAM) |

**Headline metric: output tokens/s** for the real `generate_samples` workload. Speculative
decoding is *lossless* (same output distribution), so quality is unaffected — the only
question is speed.

### Things to keep in mind while reading the results
- **Compare the draft against the target's ~3.3B *active* params**, not its 30B total (it's an
  A3B MoE). That's why we skip the 4B draft — it isn't cheaper than the target per token.
- **Batch size shrinks the gains.** Speculative decoding shines at batch 1; your real workload
  uses `num_samples` (e.g. 8) per prompt, which is closer to compute-bound. Benchmark at your
  *real* `NUM_SAMPLES`, and optionally also at 1 to see the best case.
- **Temperature matters.** Lower temperature → higher acceptance → bigger speedup. At
  `TEMPERATURE = 0` all configs produce *identical* tokens (lossless + greedy), so wall-time is
  a perfectly clean apples-to-apples comparison and an upper bound on the benefit.
- **Acceptance rate** is printed by vLLM in each worker's log (look for
  `Speculative metrics: Draft acceptance rate ...`).

### How it runs
Each config is benchmarked in its **own subprocess** ([`spec_bench_worker.py`](spec_bench_worker.py))
so the GPU is fully freed between runs. A failing config (e.g. a draft whose vocab doesn't match
the target) is reported and skipped without killing the rest.

## 1. Configuration
Edit these to match your real `generate_samples` usage and hardware.

In [ ]:
import os, sys, json, time, subprocess
from pathlib import Path

# Paths (this notebook lives in <repo>/notebooks/).
NB_DIR = Path.cwd()
REPO_ROOT = NB_DIR.parent
GEN_SCRIPT_DIR = REPO_ROOT / "kernel_gen"
WORKER = NB_DIR / "spec_bench_worker.py"
WORKDIR = NB_DIR / "spec_bench_out"
WORKDIR.mkdir(exist_ok=True)

# Models.
TARGET_MODEL = "Qwen/Qwen3-Coder-30B-A3B-Instruct-FP8"
DRAFT_MODELS = ["Qwen/Qwen3-1.7B", "Qwen/Qwen3-0.6B"]

# Workload — mirror your real generate_samples call.
DATASET_NAME = "ScalingIntelligence/KernelBench"
LEVEL = 2
PROBLEM_IDS = [0, 1, 2, 3, 4]   # a handful; more prompts = steadier averages, slower
BACKEND = "triton"
OPTION = "one_shot"

# Sampling params. NOTE: set TEMPERATURE=0 + NUM_SAMPLES=1 for the cleanest comparison
# (identical tokens across configs); use your real values (e.g. 0.8 / 8) for a realistic one.
TEMPERATURE = 0.8
NUM_SAMPLES = 8
MAX_NEW_TOKENS = 2048

# Speculative + engine params.
NUM_SPEC_TOKENS = 5          # k: draft tokens proposed per step (tune this!)
PROMPT_LOOKUP_MAX = 4        # ngram match length (max)
PROMPT_LOOKUP_MIN = 2        # ngram match length (min)
GPU_MEM_UTIL = 0.85          # lower if a draft config OOMs (the draft needs its own VRAM)
MAX_MODEL_LEN = 16384
ENFORCE_EAGER = False        # set True if spec decoding errors with CUDA graphs on your vLLM
REPEATS = 1                  # >1 averages out noise (each repeat re-runs all prompts)

print("repo root :", REPO_ROOT)
print("worker    :", WORKER, "(exists)" if WORKER.exists() else "(MISSING!)")
print("target    :", TARGET_MODEL)
print("drafts    :", DRAFT_MODELS)

## 2. Vocab / tokenizer compatibility check (CPU)
vLLM's **draft-model** method requires the draft and target to share the same vocabulary.
Qwen3-Coder added tool-calling special tokens, so this is *not* guaranteed against the Qwen3
dense models. If a draft's `vocab_size` differs from the target, its draft config will fail to
load (the benchmark will just skip it). `ngram` is unaffected.

In [ ]:
from transformers import AutoConfig, AutoTokenizer

def vocab_info(model_id):
    c = AutoConfig.from_pretrained(model_id, trust_remote_code=True)
    t = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
    return getattr(c, "vocab_size", None), len(t)

tgt_cfg_vocab, tgt_tok_vocab = vocab_info(TARGET_MODEL)
print(f"{TARGET_MODEL}\n    config_vocab={tgt_cfg_vocab}  tok_vocab={tgt_tok_vocab}\n")
for dm in DRAFT_MODELS:
    cv, tv = vocab_info(dm)
    ok = (cv == tgt_cfg_vocab)
    print(f"{dm}\n    config_vocab={cv}  tok_vocab={tv}  ->  {'OK (matches target)' if ok else 'MISMATCH: draft method will likely fail'}\n")

## 3. Build the prompts (CPU, no GPU)
Uses the same `get_prompt_for_backend` path as `generate_kernels_samples.py`, so the prompts
are identical to production. Saved to disk and reused by every config.

In [ ]:
sys.path.insert(0, str(GEN_SCRIPT_DIR))
sys.path.insert(0, str(GEN_SCRIPT_DIR / "KernelBench" / "src"))
from datasets import load_dataset
from kernelbench.prompt_constructor_toml import get_prompt_for_backend

ds = load_dataset(DATASET_NAME, split=f"level_{LEVEL}")
prompts = []
for pid in PROBLEM_IDS:
    ref = ds[pid]["code"]
    prompts.append(get_prompt_for_backend(
        ref_arch_src=ref, backend=BACKEND, option=OPTION,
        include_hardware=False, gpu_name=None,
    ))

PROMPTS_FILE = WORKDIR / "prompts.json"
PROMPTS_FILE.write_text(json.dumps(prompts))
print(f"Built {len(prompts)} prompts (avg {sum(len(p) for p in prompts)//len(prompts)} chars) -> {PROMPTS_FILE}")

## 4. Define the configs and the per-config runner
Each config is run by `spec_bench_worker.py` in a fresh subprocess. We do **not** capture the
child's output, so you'll see vLLM's logs live — including the `Speculative metrics: Draft
acceptance rate ...` lines for the draft/ngram runs.

In [ ]:
def make_configs():
    common = dict(
        target_model=TARGET_MODEL,
        gen_script_dir=str(GEN_SCRIPT_DIR),
        prompts_file=str(PROMPTS_FILE),
        temperature=TEMPERATURE,
        num_samples=NUM_SAMPLES,
        max_new_tokens=MAX_NEW_TOKENS,
        gpu_memory_utilization=GPU_MEM_UTIL,
        max_model_len=MAX_MODEL_LEN,
        enforce_eager=ENFORCE_EAGER,
        repeats=REPEATS,
        num_speculative_tokens=NUM_SPEC_TOKENS,
        prompt_lookup_max=PROMPT_LOOKUP_MAX,
        prompt_lookup_min=PROMPT_LOOKUP_MIN,
    )
    configs = [{**common, "label": "baseline (no spec)", "spec_method": "none"}]
    for dm in DRAFT_MODELS:
        configs.append({**common, "label": f"draft {dm.split('/')[-1]}",
                        "spec_method": "draft", "draft_model": dm})
    configs.append({**common, "label": f"ngram (k={NUM_SPEC_TOKENS})", "spec_method": "ngram"})
    return configs


def _slug(label):
    return label.replace(' ', '_').replace('/', '_').replace('(', '').replace(')', '').replace('=', '')


def run_config(cfg):
    slug = _slug(cfg["label"])
    cfg_file = WORKDIR / f"cfg_{slug}.json"
    out_file = WORKDIR / f"out_{slug}.json"
    cfg_file.write_text(json.dumps(cfg, indent=2))
    print(f"\n{'=' * 70}\n=== {cfg['label']} ===\n{'=' * 70}", flush=True)
    t0 = time.perf_counter()
    proc = subprocess.run(
        [sys.executable, str(WORKER), "--config-file", str(cfg_file), "--out-file", str(out_file)],
        cwd=str(REPO_ROOT),
    )
    wall = time.perf_counter() - t0
    if proc.returncode != 0 or not out_file.exists():
        print(f"[FAILED] {cfg['label']} (returncode {proc.returncode}) — see log above")
        return None
    res = json.loads(out_file.read_text())
    res["subprocess_wall_s"] = wall
    print(f"[OK] {cfg['label']}: {res['tokens_per_s_mean']:.1f} tok/s")
    return res

## 5. Run the benchmark
This loads each model in turn (slow on first run — models download + compile). Watch the logs
for the draft-acceptance metrics.

In [ ]:
results = []
for cfg in make_configs():
    r = run_config(cfg)
    if r is not None:
        results.append(r)

print(f"\nCollected {len(results)} successful configs.")

## 6. Results: speedup vs baseline
`speedup > 1` means the proposer actually helped *for this workload at this batch size and
temperature*. Anything `<= 1` means it cost more than it saved — exactly the thing we wanted to
find out before committing to it.

In [ ]:
import pandas as pd

df = pd.DataFrame(results)
base_row = df[df.spec_method == "none"]
assert len(base_row) == 1, "need exactly one baseline run to normalize against"
base_tps = base_row["tokens_per_s_mean"].iloc[0]
df["speedup_vs_baseline"] = df["tokens_per_s_mean"] / base_tps

show = (df[["label", "tokens_per_s_mean", "wall_s_mean", "speedup_vs_baseline", "load_s"]]
        .sort_values("tokens_per_s_mean", ascending=False)
        .reset_index(drop=True))
show.round({"tokens_per_s_mean": 1, "wall_s_mean": 1, "speedup_vs_baseline": 3, "load_s": 1})

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
order = df.sort_values("tokens_per_s_mean")
ax1.barh(order["label"], order["tokens_per_s_mean"])
ax1.set_xlabel("output tokens / s")
ax1.set_title(f"Throughput (T={TEMPERATURE}, n={NUM_SAMPLES})")

ax2.barh(order["label"], order["speedup_vs_baseline"])
ax2.axvline(1.0, color="red", linestyle="--", label="baseline")
ax2.set_xlabel("speedup vs baseline (>1 = faster)")
ax2.set_title("Speedup")
ax2.legend()
plt.tight_layout()
plt.show()

## 7. Interpreting & next steps

- **If no config beats `1.0`:** at your batch size (`NUM_SAMPLES`) the target is compute-bound and
  speculation doesn't pay off. Re-run with `NUM_SAMPLES = 1` (and `TEMPERATURE = 0`) to see the
  best-case / latency-bound number — if it doesn't help even there, drop speculative decoding for
  this workload.
- **Tune `NUM_SPEC_TOKENS` (k).** Sweep e.g. `[3, 5, 7]`. Higher k helps when acceptance is high,
  hurts when it's low. Read the `Draft acceptance rate` from the worker logs to guide this.
- **1.7B vs 0.6B:** 1.7B should have higher acceptance, 0.6B is cheaper per draft token. The winner
  is empirical — that's the whole point of this notebook.
- **`ngram` is the zero-VRAM baseline.** It exploits the heavy verbatim copying from the prompt and
  costs no extra memory; a draft model only wins if it also accelerates the *novel* kernel code.
- **Clean measurement mode:** set `TEMPERATURE = 0`, `NUM_SAMPLES = 1`. Then every config emits
  identical tokens (lossless + greedy), so `wall_s_mean` is a direct, noise-free comparison.
- Quality never changes — speculative decoding is output-equivalent to the target. You're only ever
  trading on speed.